In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))


# 02 Feature Engineering — SENTINEL Phase 2

Past-only feature store. Each split (`train` / `valid` / `test`) is transformed **separately** so no future or cross-split statistics leak.
Outputs (canonical, train-based) in `artifacts/`: `transaction_features.csv`, `feature_dictionary.json`, `feature_manifest.json`, `isolation_forest_features.json`, `feature_quality_report.md`, `graph_nodes.csv`, `graph_edges.csv`.
No model training in this notebook.

In [ ]:
import json
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
log = logging.getLogger('phase2')

from src import config
from src.feature_store import load_feat_config

ARTIFACTS_DIR = config.ARTIFACTS_DIR
feat_cfg = load_feat_config()  # reads config/features.yaml
log.info('feat_cfg keys=%s', sorted(feat_cfg.keys()))
print(json.dumps({k: feat_cfg[k] for k in sorted(feat_cfg.keys())}, indent=1, default=str))

# (1) Load splits produced by 01 (chronological, no shuffle)
train_df = pd.read_csv(ARTIFACTS_DIR / 'train.csv', parse_dates=['date'])
valid_df = pd.read_csv(ARTIFACTS_DIR / 'valid.csv', parse_dates=['date'])
test_df = pd.read_csv(ARTIFACTS_DIR / 'test.csv', parse_dates=['date'])
log.info('train=%s valid=%s test=%s', train_df.shape, valid_df.shape, test_df.shape)
print(train_df.shape, valid_df.shape, test_df.shape)
print(train_df.columns.tolist())


## Build feature store per split (past-only, no cross-split leakage)

`build_feature_store` applies `add_features` + `add_phase2_features` (shifted expanding means, prior-window counts only) and writes the 7 artifacts. We run it on each split separately; the final write is `train` so `artifacts/` holds the canonical train-based store for Phase 3.

In [ ]:
from src.feature_store import build_feature_store

# (2) Run on each split SEPARATELY — past-only within that split, never pooled.
# Order: valid, test, then train last so canonical artifacts/ files are train-based.
valid_feat = build_feature_store(valid_df, out_dir=ARTIFACTS_DIR, cfg=feat_cfg)
test_feat = build_feature_store(test_df, out_dir=ARTIFACTS_DIR, cfg=feat_cfg)
train_feat = build_feature_store(train_df, out_dir=ARTIFACTS_DIR, cfg=feat_cfg)

log.info('built train=%s valid=%s test=%s', train_feat.shape, valid_feat.shape, test_feat.shape)
print(train_feat.shape, valid_feat.shape, test_feat.shape)


In [ ]:
import numpy as np

# (3) Leakage checks — monotonic timestamps + no future stats
# 3a. Timestamps must be non-decreasing per account (feature builder sorts past-only)
for name, feat in [('train', train_feat), ('valid', valid_feat), ('test', test_feat)]:
    d = pd.to_datetime(feat['date'])
    bad = feat.assign(_d=d).groupby('account_id')['_d'].apply(lambda s: bool((s.diff().dropna() < pd.Timedelta(0)).any()))
    assert not bad.any(), f'{name}: non-monotonic timestamps in {bad[bad].index.tolist()[:3]}'
    log.info('%s monotonic timestamps OK (%d accounts)', name, feat['account_id'].nunique())
print('monotonic timestamps per account: OK')

# 3b. amount_ratio must use SHIFTED expanding mean (no current-row leakage)
counts = train_feat['account_id'].value_counts()
sample_acct = counts[counts >= 3].index[0]
sub = train_feat[train_feat['account_id'] == sample_acct].sort_values('date', kind='mergesort').reset_index(drop=True)
manual_avg = sub['abs_amount'].expanding().mean().shift(1).fillna(train_feat['abs_amount'].median())
assert np.allclose(sub['customer_avg_amount'].values, manual_avg.values, rtol=1e-6, atol=1e-6), 'customer_avg_amount is not shifted expanding mean'
manual_ratio = sub['abs_amount'] / (manual_avg + 1)
assert np.allclose(sub['amount_ratio'].values, manual_ratio.values, rtol=1e-6, atol=1e-6), 'amount_ratio leaks future/current stats'
# First txn of account must fall back to global median (no history)
assert sub.loc[0, 'customer_avg_amount'] == manual_avg.iloc[0]
log.info('sample account %s (%d txns): shifted expanding mean verified', sample_acct, len(sub))
print(f'sample account {sample_acct}: amount_ratio == abs_amount/(shifted_expanding_mean+1): OK')
print('LEAKAGE CHECKS PASSED')


In [ ]:
import json
from pathlib import Path

# (4) Feature preview + quality summary
print(train_feat.head(3).to_string())
print(train_feat[['abs_amount', 'amount_ratio', 'amount_zscore', 'txns_last_day', 'txns_last_7d', 'new_beneficiary', 'passthrough_ratio']].describe().to_string())

report = (ARTIFACTS_DIR / 'feature_quality_report.md').read_text().splitlines()
print('\n'.join(report[:12]))
print('...')
if_cands = json.loads((ARTIFACTS_DIR / 'isolation_forest_features.json').read_text())
manifest = json.loads((ARTIFACTS_DIR / 'feature_manifest.json').read_text())
print('IF candidates:', if_cands)
print('manifest groups:', {k: len(v) for k, v in manifest.items()})

# Final message: list all generated artifacts
expected = ['transaction_features.csv', 'feature_dictionary.json', 'feature_manifest.json', 'isolation_forest_features.json', 'feature_quality_report.md', 'graph_nodes.csv', 'graph_edges.csv']
for f in expected:
    p = ARTIFACTS_DIR / f
    print(('OK  ' if p.exists() else 'MISS') + f' artifacts/{f} ({p.stat().st_size if p.exists() else 0} bytes)')
    assert p.exists(), f'missing artifacts/{f}'
print('PHASE 2 COMPLETE: all 7 artifacts in artifacts/ | NEXT PHASE 03_rules_engine_and_isolation_forest.ipynb')
